In [1]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn

from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.linear_model import ElasticNet
from urllib.parse import urlparse

import logging

In [2]:
logging.basicConfig(level=logging.WARN)
logger = logging.getLogger(__name__)

In [3]:
def eval_metrics(actual, pred):
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mae = mean_absolute_error(actual, pred)
    r2 = r2_score(actual, pred)
    return rmse, mae, r2

In [4]:
data = pd.read_csv('winequality-red.csv', sep=';')
data.shape

(1599, 12)

In [5]:
train, test = train_test_split(data)
x_train = train.drop(["quality"], axis=1)
x_test = test.drop(["quality"], axis=1)
y_train = train[['quality']]
y_test = test[['quality']]

alpha, l1_ratio = 0.5, 0.5

In [6]:
run_id = 0
with mlflow.start_run() as run:
    run_id = run.info.run_id
    lr = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, random_state=42)
    lr.fit(x_train, y_train)
    
    pred = lr.predict(x_test)
    
    (rmse, mae, r2) = eval_metrics(y_test, pred)
    
    print("Elasticnet model (alpha={:f}, l1_ratio={:f}):".format(alpha, l1_ratio))
    print("  RMSE: %s" % rmse)
    print("  MAE: %s" % mae)
    print("  R2: %s" % r2)
    
    mlflow.log_params({
        "alpha": alpha,
        "l1_ratio": l1_ratio
    })
    
    mlflow.log_metrics({
        "rmse": rmse,
        "mae": mae,
        "r2": r2
    })
    
    tracking_url_store = urlparse(mlflow.get_tracking_uri()).scheme
    
    if tracking_url_store != 'file':
        mlflow.sklearn.log_model(lr, "model", registered_model_name="ElasticNetWineModel")
    else:
        mlflow.sklearn.log_model(lr, "model")

Elasticnet model (alpha=0.500000, l1_ratio=0.500000):
  RMSE: 0.7754127588025348
  MAE: 0.6320813787512635
  R2: 0.11786246110070742


In [7]:
mlflow.end_run()